# Get Seasonal Wetland Fractions

In [ ]:
import sys
dir_git = "/Users/usuario/git/sisepuede"
if dir_git not in sys.path:
    sys.path.append(dir_git)
    
import importlib
import matplotlib.pyplot as plt
import numpy as np
import os, os.path
import pandas as pd
import pathlib
import sisepuede.core.support_classes as sc
import sisepuede.manager.sisepuede_file_structure as sfs
import sisepuede.manager.sisepuede_models as sm
import sisepuede.utilities._toolbox as sf
import warnings
warnings.filterwarnings("ignore")

import utils.common_data_needs as cdn



In [75]:
importlib.reload(cdn)
dict_ssp = cdn._setup_sisepuede_elements()

matt = dict_ssp.get("model_attributes", )
models = dict_ssp.get("models", )
regions = dict_ssp.get("regions", )
time_periods = dict_ssp.get("time_periods", )

In [7]:
##  SET SOME GLOBALS FOR ANALYSIS

# model variables
_MODVAR_AREA = matt.get_variable("Land Use Area")
_MODVAR_SEASONAL_FRAC = matt.get_variable("Seasonal Wetland Fraction")

# set some derivative file names
_FILE_NAME_SEASONAL_FRAC = cdn.file_name_from_variable(_MODVAR_SEASONAL_FRAC, )

# units managers
_UM_AREA = matt.get_unit("area")


# get base data, ignoring the inputs constructed herein
df_uganda = cdn._build_from_outputs(
    (
        min(time_periods.all_years),
        max(time_periods.all_years)
    ),
    fns_exclude = [
        _FILE_NAME_SEASONAL_FRAC,
    ], 
    force_complete_build = True,
    merge_type = "outer",
    print_info = False,
    stop_on_error = True, 
)

_DF_OUT = models.model_afolu(df_uganda, )

# get base data frames from base dataset; we'll overwrite data below, then export it
_DF_AREA = cdn.get_variable_df_for_overwrite(
    _DF_OUT, 
    _MODVAR_AREA,
)


# Get MWE Wetlands Department data
- Data are sourced directly from Wetlands GIS department
- Crosswalk dictionaries are representations of:
    - _classification 2015.docx_
    - _2021 CLASSIFICATION CODE.docx_

In [95]:
"""
ONLY RUN THIS IF THE CSVs ARE OUTDATED OR UNAVAILABLE

fn2015 = "Wetlands 2015.xlsx"
fn2021 = "Wetlands2021.xlsx"

path_data_2015_xlsx = path_wetlands.joinpath(fn2015, )
path_data_2021_xlsx = path_wetlands.joinpath(fn2021, )

# this is slow
df2015 = pd.read_excel(path_data_2015_xlsx, )
df2021 = pd.read_excel(path_data_2021_xlsx, )

# set outputs
path_data_2015 = path_wetlands.joinpath(fn2015.replace(".xlsx", ".csv"), )
path_data_2021 = path_wetlands.joinpath(fn2021.replace(".xlsx", ".csv"), )
sf._write_csv(df2015, path_data_2015, )
sf._write_csv(df2021, path_data_2021, )
"""


In [125]:
path_wetlands = cdn._PATH_INPUTS.joinpath("mwe", "wetlands_department_meeting_202604")
path_data_2015 = path_wetlands.joinpath("Wetlands 2015.csv")
path_data_2021 = path_wetlands.joinpath("Wetlands2021.csv")



###############################
#    SOME GLOBAL VARIABLES    #
###############################

# class mapping for 2015
_DICT_CLASS_2015 = {
    6: "shrublands",
    7: "grasslands",
    9: "croplands",
    10: "croplands",
    13: "other",
}

# class mapping for 2021
_DICT_CLASS_2021 = {
    2: "grasslands",
    5: "croplands",
    7: "shrublands",
    9: "croplands",
}


#
_FIELD_W2015_AREA = "Area_Ha"
_FIELD_W2015_CLASS = "Class_2015"
_FIELD_W2015_SOIL = "Soil_moist"

# 2021 fields
_FIELD_W2021_AREA = "Area_Ha"
_FIELD_W2021_CODE = "gridcode"
_FIELD_W2021_SEASONAL = "Seasonalit"
_FIELD_W2021_STATUS = "Status"

# keys/sheets
_KEY_2015 = "Wetlands_2015"
_KEY_2021 = "Wetlands2021"




#######################
#    GET FUNCTIONS    #
#######################

def get_data_2015(
    path: pathlib.Path, 
) -> pd.DataFrame:
    """Get the 2015 seasonal wetland areas by SSP land use type
    """

    # read and note area
    df_2015 = pd.read_csv(path, )

    area_total = df_2015[_FIELD_W2015_AREA].sum()
    print(f"Total area in 2015 data before filtering: {area_total}")
    

    # filter
    df_2015 = (
        df_2015[
            df_2015[_FIELD_W2015_SOIL].isin(["Seasonally wet"])
        ]
        .get([
            _FIELD_W2015_AREA,
            _FIELD_W2015_CLASS,
        ])
    )

    # replace codes for aggregation
    df_2015[_FIELD_W2015_CLASS] = (
        df_2015[_FIELD_W2015_CLASS]
        .replace(_DICT_CLASS_2015, )
    )

    
    ##  AGGREGATE AND BUILD DICTIONARY
    
    df_2015 = (
        df_2015
        .groupby(_FIELD_W2015_CLASS)
        .sum()
        .reset_index()
    )

    # filter
    df_2015 = sf.build_dict(
        df_2015[
            df_2015[_FIELD_W2015_CLASS].isin(
                _DICT_CLASS_2015.values()
            )
        ]
        .get(
            [
                _FIELD_W2015_CLASS,
                _FIELD_W2015_AREA
            ]
        )
    )

    return df_2015



def get_data_2021(
    path: pathlib.Path, 
) -> Dict[str, float]:
    """Get the 2021 seasonal wetland areas by SSP land use type
    """

    # read and note total area
    df_2021 = pd.read_csv(path, )
    
    area_total = df_2021[_FIELD_W2021_AREA].sum()
    print(f"Total area in 2021 data before filtering: {area_total}")

    
    # filter
    df_2021 = (
        df_2021[
            df_2021[_FIELD_W2021_SEASONAL].isin(["Seasonal"])
            & df_2021[_FIELD_W2021_STATUS].isin(["Intact"])
        ]
        .drop(
            columns = [
                _FIELD_W2021_SEASONAL,
                _FIELD_W2021_STATUS,
            ]
        )
    )

    # replace codes for aggregation
    df_2021[_FIELD_W2021_CODE] = (
        df_2021[_FIELD_W2021_CODE]
        .replace(_DICT_CLASS_2021, )
    )


    ##  AGGREGATE AND BUILD DICTIONARY
    
    # aggregate
    df_2021 = (
        df_2021
        .groupby(_FIELD_W2021_CODE)
        .sum()
        .reset_index()
    )

    # filter
    df_2021 = sf.build_dict(
        df_2021[
            df_2021[_FIELD_W2021_CODE].isin(
                _DICT_CLASS_2021.values()
            )
        ]
        .get(
            [
                _FIELD_W2021_CODE,
                _FIELD_W2021_AREA
            ]
        )
    )

    return df_2021


# The 2021 data is incomplete--use only numbers from 2015
- 2015 accounts for about 2.77 million ha of seasonal wetlands, in line with MWE's comments
- 2021 accounts for only ≈0.188 million ha

In [135]:
_DICT_AREA_SW_2015 = get_data_2015(path_data_2015, )

Total area in 2015 data before filtering: 2774119.681997525


In [136]:
_DICT_AREA_SW_2021 = get_data_2021(path_data_2021, )

Total area in 2021 data before filtering: 188455.94955100637


In [174]:
cdn.spawn_years_space_df((2015, 2100, ))

,year
0,2015
1,2016
2,2017
3,2018
4,2019
...,...
80,2095
81,2096
82,2097
83,2098


In [175]:
# area of land in 2015
_DICT_AREA_LNDU_2015 = (
    _DF_AREA[
        _DF_AREA[time_periods.field_year].isin([2015])
    ]
    .iloc[0]
    .to_dict()
)

# get categories before iterating over to get fractions
cat_grss = models.model_afolu.cat_lndu_grss
cat_pstr = models.model_afolu.cat_lndu_pstr
cats_lndu = matt.get_variable_categories(_MODVAR_SEASONAL_FRAC, )

dict_fracs = {}

for cat in cats_lndu:
    # aggregate pastures manually with grasslands
    if cat == cat_pstr: continue
        
    # try getting area of seasonal wetlands
    area_seasonal = _DICT_AREA_2015.get(cat, )
    if area_seasonal is None: continue

    # if at grasslands, include pastures in total area since seasonal wetlands will inclue those
    cats = [cat, cat_pstr] if cat == cat_grss else [cat]
    fields_lndu_area = _MODVAR_AREA.build_fields(category_restrictions = cats, )
    if fields_lndu_area is None: continue

    # try retrieving area of land
    area_land = sum(
        [
            _DICT_AREA_LNDU_2015.get(x, 0, )
            for x in fields_lndu_area
        ]
    )
    if area_land == 0: continue

    # if we made it this far, set the field and add to the dictionary
    val = min(area_seasonal/area_land, 1.0, )
    
    fields = _MODVAR_SEASONAL_FRAC.build_fields(
        category_restrictions = cats,
    )
    
    dict_fracs.update(dict((field, val) for field in fields))
    
dict_fracs


{'frac_lndu_seasonal_wetland_croplands': 0.08172368542205848,
 'frac_lndu_seasonal_wetland_grasslands': 0.11610114258166922,
 'frac_lndu_seasonal_wetland_pastures': 0.11610114258166922,
 'frac_lndu_seasonal_wetland_other': 0.027984716474867367,
 'frac_lndu_seasonal_wetland_shrublands': 0.15973242358324774}

# Convert to an output DataFrame

In [186]:
df_out = cdn.spawn_years_space_df((2015, 2101))
for k, v in dict_fracs.items():
    df_out[k] = v

df_out = _MODVAR_SEASONAL_FRAC.get_from_dataframe(
    df_out,
    extraction_logic = "any_fill",
    fields_additional = [time_periods.field_year],
    fill_value = 0.0,
)


sf._write_csv(
    df_out,
    cdn._PATH_OUTPUTS.joinpath(_FILE_NAME_SEASONAL_FRAC),
)

True

In [191]:
_MODVAR_AREA.get_from_dataframe(_DF_AREA).sum(axis =1 )*0.078

0     1.884090e+06
1     1.884090e+06
2     1.884090e+06
3     1.884090e+06
4     1.884090e+06
5     1.884090e+06
6     1.884090e+06
7     1.884090e+06
8     1.884090e+06
9     1.884090e+06
10    1.884090e+06
11    1.884090e+06
12    1.884090e+06
13    1.884090e+06
14    1.884090e+06
15    1.884090e+06
16    1.884090e+06
17    1.884090e+06
18    1.884090e+06
19    1.884090e+06
20    1.884090e+06
21    1.884090e+06
22    1.884090e+06
23    1.884090e+06
24    1.884090e+06
25    1.884090e+06
26    1.884090e+06
27    1.884090e+06
28    1.884090e+06
29    1.884090e+06
30    1.884090e+06
31    1.884090e+06
32    1.884090e+06
33    1.884090e+06
34    1.884090e+06
35    1.884090e+06
36    1.884090e+06
37    1.884090e+06
38    1.884090e+06
39    1.884090e+06
40    1.884090e+06
41    1.884090e+06
42    1.884090e+06
43    1.884090e+06
44    1.884090e+06
45    1.884090e+06
46    1.884090e+06
47    1.884090e+06
48    1.884090e+06
49    1.884090e+06
50    1.884090e+06
51    1.884090e+06
52    1.8840